In [28]:
from __future__ import annotations

import operator
import os
import re
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openrouter import ChatOpenRouter

from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults

In [29]:
import sys
import os
import httpx
import anyio
import ipykernel.iostream
from dotenv import load_dotenv

# Fix Jupyter OutStream fileno error for Windows MCP subprocesses
ipykernel.iostream.OutStream.fileno = lambda self: sys.__stderr__.fileno()
sys.stderr = sys.__stderr__

load_dotenv()

import langchain
langchain.debug = False

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.client.session import ClientSession
from mcp.types import JSONRPCMessage

# Add Streamable HTTP transport support to MultiServerMCPClient
async def _connect_via_http(self, server_name: str, *, url: str, headers: dict = None, **kwargs):
    req_headers = dict(headers or {})
    req_headers['Content-Type'] = 'application/json'
    read_stream_writer, read_stream = anyio.create_memory_object_stream(0)
    write_stream, write_stream_reader = anyio.create_memory_object_stream(0)
    session_id = None

    client = await self.exit_stack.enter_async_context(httpx.AsyncClient(headers=req_headers, timeout=30.0))
    tg = await self.exit_stack.enter_async_context(anyio.create_task_group())

    async def reader():
        nonlocal session_id
        async with write_stream_reader:
            async for message in write_stream_reader:
                post_headers = dict(req_headers)
                if session_id:
                    post_headers['Mcp-Session-Id'] = session_id
                resp = await client.post(url, json=message.model_dump(by_alias=True, mode='json', exclude_none=True), headers=post_headers)
                if 'mcp-session-id' in resp.headers:
                    session_id = resp.headers['mcp-session-id']
                for line in resp.text.splitlines():
                    if line.startswith('data: '):
                        data_str = line[6:].strip()
                        if data_str:
                            msg = JSONRPCMessage.model_validate_json(data_str)
                            await read_stream_writer.send(msg)

    tg.start_soon(reader)
    session = await self.exit_stack.enter_async_context(ClientSession(read_stream, write_stream))
    await self._initialize_session_and_load_tools(server_name, session)

MultiServerMCPClient.connect_to_server_via_http = _connect_via_http

async def _patched_aenter(self):
    try:
        connections = self.connections or {}
        for server_name, connection in connections.items():
            connection_dict = connection.copy()
            transport = connection_dict.pop('transport')
            if transport == 'http':
                await self.connect_to_server_via_http(server_name, **connection_dict)
            elif transport == 'stdio':
                await self.connect_to_server_via_stdio(server_name, **connection_dict)
            elif transport == 'sse':
                await self.connect_to_server_via_sse(server_name, **connection_dict)
        return self
    except Exception:
        await self.exit_stack.aclose()
        raise

MultiServerMCPClient.__aenter__ = _patched_aenter

github_token = os.environ.get('GITHUB_ACCESS_TOKEN')

# Initialize MCP client with GitHub Copilot HTTP MCP server
mcp_client = MultiServerMCPClient(
    {
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp",
            "headers": {
                "Authorization": f"Bearer {github_token}"
            },
        }
    }
)

await mcp_client.__aenter__()
mcp_tools = mcp_client.get_tools()

print(f"\n✅ Successfully loaded {len(mcp_tools)} total MCP tools:")



✅ Successfully loaded 44 total MCP tools:


In [50]:
from langchain_community.document_loaders import GithubFileLoader
import mimetypes

def is_text_file(file_path: str) -> bool:
    mime_type, _ = mimetypes.guess_type(file_path)

    if mime_type is None:
        return False

    return (
        mime_type.startswith("text/")
        or mime_type in {
            "application/json",
            "application/javascript",
            "application/xml",
            "application/x-python",
        }
    )

def convert_github_url_to_repo_id(github_url: str) -> str:
    """Convert a GitHub repository URL to owner/repo format."""

    cleaned = github_url.strip().rstrip("/")

    # Remove protocol
    cleaned = cleaned.replace("https://", "").replace("http://", "")

    # Remove .git suffix
    cleaned = cleaned.removesuffix(".git")

    parts = cleaned.split("/")

    if len(parts) < 3 or parts[0].lower() != "github.com":
        raise ValueError("Invalid GitHub repository URL")

    owner = parts[1]
    repo = parts[2]

    return f"{owner}/{repo}"

In [51]:
def github_loader(repo_url, branch="main"):
    """Load GitHub repository files"""
    repo_id = convert_github_url_to_repo_id(repo_url)
    loader = GithubFileLoader(
        repo=repo_id,
        branch=branch,
        file_filter= is_text_file,
        access_token=os.environ.get("GITHUB_ACCESS_TOKEN"),
    )
    docs = loader.load()
    full_text = ""
    for i, doc in enumerate(docs, start=1):
        file_name = doc.metadata.get("source", f"file_{i}")
        full_text += f"\n\n===== FILE {i}: {file_name} =====\n"
        full_text += doc.page_content
    return full_text

In [52]:
result = github_loader("https://github.com/M-Azfar9/Model-Context-Protocol")

In [53]:
result

'\n\n===== FILE 1: https://api.github.com/M-Azfar9/Model-Context-Protocol/blob/main/Local Servers/browser-history-mcp/README.md =====\n# Browser History MCP\n\nA powerful [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) server that provides read-only access to your local browser history and bookmarks. This allows AI assistants to help you find information you\'ve seen before, summarize your browsing habits, or look up bookmarks stored in **Google Chrome** and **Mozilla Firefox**.\n\n## Features\n\n- **Multi-Browser Support**: Automatically detects and queries Chrome and Firefox profiles.\n- **Safe Read-Access**: Copies database files to temporary locations to avoid "database is locked" errors while your browser is open.\n- **Privacy First**: All processing is local. No data is transmitted externally; it only provides the data to your local AI assistant.\n- **Deep Search**: Search history and bookmarks by keyword, URL, or folder name.\n- **Insights**: Identify top domain

('text/plain', None)
('text/x-python', None)
('image/png', None)
('application/json', None)


In [37]:
result = await get_file_contents_tool.ainvoke({
    "owner": "github",
    "repo": "github-mcp-server",
    "path": "README.md"
})

result

'successfully downloaded text file (SHA: 6585ab30f6d30c81970084cd8a15b750cc57b8f5)'

In [40]:
print("TYPE:")
print(type(result))

print("\nDIR:")
print(dir(result))

print("\nREPR:")
print(repr(result))

TYPE:
<class 'str'>

DIR:
['__add__', '__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mod__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__rmod__', '__rmul__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'capitalize', 'casefold', 'center', 'count', 'encode', 'endswith', 'expandtabs', 'find', 'format', 'format_map', 'index', 'isalnum', 'isalpha', 'isascii', 'isdecimal', 'isdigit', 'isidentifier', 'islower', 'isnumeric', 'isprintable', 'isspace', 'istitle', 'isupper', 'join', 'ljust', 'lower', 'lstrip', 'maketrans', 'partition', 'removeprefix', 'removesuffix', 'replace', 'rfind', 'rindex', 'rjust', 'rpartition', 'rsplit', 'rstrip', 'split', 'splitlines', 'startswith', 'strip', 'swapcase', 'title', 'translate', 'upper', 'z

In [3]:
from langchain_cohere import ChatCohere
from langchain_core.messages import HumanMessage

# ============================================
# 1. Your Cohere API Key
# ============================================
COHERE_API_KEY = ""


# ============================================
# 2. Create the Cohere Chat Model
# ============================================
llm = ChatCohere(
    cohere_api_key=COHERE_API_KEY,
    model="command-a-03-2025",
    temperature=0
)


# ============================================
# 3. Test the model
# ============================================
message = HumanMessage(
    content="Hi"
)

response = llm.invoke([message])


# ============================================
# 4. Print the response
# ============================================================
print("MODEL RESPONSE:")

for chunk in response:
    if chunk.text:
        print(chunk.text, end="", flush=True)

print()

# ============================================
# 5. Inspect metadata
# ============================================
print("\n" + "=" * 60)
print("RESPONSE TYPE:")
print(type(response))

print("\n" + "=" * 60)
print("RESPONSE METADATA:")
print(response.response_metadata)

MODEL RESPONSE:


AttributeError: 'tuple' object has no attribute 'text'

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

# ============================================================
# 1. Google Gemini API Key
# ============================================================
GOOGLE_API_KEY = ""


# ============================================================
# 2. Create Gemini 3.7 Flash model
# ============================================================
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=GOOGLE_API_KEY,
    thinking_level="medium",
    streaming=True, 
)


# ============================================================
# 3. Stream the response
# ============================================================
print("MODEL RESPONSE (STREAMING):")
print("-" * 40)

full_response = ""
metadata = None
usage_metadata = None

# Use stream() instead of invoke() for streaming
for chunk in llm.stream("What is RAG?"):
    # Extract the content from the chunk
    if hasattr(chunk, 'content'):
        content = chunk.content[0]['text']
        print(content, end="", flush=True)  # Print without newline for real-time effect

MODEL RESPONSE (STREAMING):
----------------------------------------
**RAG** stands for **Retrieval-Augmented Generation**. 

In simple terms, RAG is a technique used in artificial intelligence to make Large Language Models (like GPT-4) smarter, more accurate, and up-to-date by allowing them to **look up information from an external database** before answering a question.

Here is the best analogy to understand it:
* **Without RAG (Closed-Book Exam):** The AI has to answer your question purely from its memory (what it learned during training). If it doesn't know the answer, or if the information is too new, it might guess or make things up (hallucinate).
* **With RAG (Open-Book Exam):** Before answering, the AI is given a search engine. It searches a specific library of documents for the right information, reads the relevant pages, and then writes a precise answer based on what it found.

---

### Why do we need RAG?
Standard LLMs have three major limitations:
1. **Knowledge Cutoff:** 

IndexError: list index out of range

In [ ]:
import langchain
import langchain_core
import langchain_google_genai

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)
print("LangChain Google GenAI:", langchain_google_genai.__version__)

LangChain: 1.3.18
LangChain Core: 1.6.1
LangChain Google GenAI: 4.4.0


In [9]:
import importlib.metadata

packages = ['langchain', 'langchain-core', 'langchain-google-genai']
for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package}: {version}")
    except:
        print(f"{package}: NOT INSTALLED")

langchain: 1.3.18
langchain-core: 1.6.1
langchain-google-genai: 4.4.0
